In [50]:
import pandas as pd
from pathlib import Path
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread_formatting import *

In [51]:
DIR_PATH = Path("Consolidated_Schedule_of_Investments/2026-03-31/")
CONSOLIDATED_CSV_PATH = Path(DIR_PATH, "outputs/ea0289743-01_ncsr.csv")
ANALYSIS_PATH = Path(DIR_PATH, "analysis")
ANALYSIS_PATH.mkdir(exist_ok=True)

DIR_PATH.exists(), CONSOLIDATED_CSV_PATH.exists(), ANALYSIS_PATH.exists()

(True, True, True)

In [52]:
df_investment = pd.read_csv(CONSOLIDATED_CSV_PATH)
df_investment.info()

<class 'pandas.DataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   portfolio_company         281 non-null    str    
 1   initial_acquisition_date  279 non-null    str    
 2   geographic_region         280 non-null    str    
 3   cost                      280 non-null    float64
 4   fair_value                280 non-null    float64
 5   asset_class               281 non-null    str    
 6   part                      281 non-null    str    
 7   table_index               281 non-null    int64  
 8   row_index                 281 non-null    int64  
 9   shares_units              34 non-null     float64
 10  principal_amount          2 non-null      float64
dtypes: float64(4), int64(2), str(5)
memory usage: 24.3 KB


In [53]:
df_investment[df_investment["portfolio_company"].str.contains("Platte River", na=False)]

,portfolio_company,initial_acquisition_date,geographic_region,cost,fair_value,asset_class,part,table_index,row_index,shares_units,principal_amount
139,"Platte River Equity III, L.P.",9/30/2024,North America,456897.0,NaN,Secondary Fund Investments — 40.4%,ea0289743-01_ncsr,7,20,NaN,NaN


In [54]:
df_investment.head(275)

,portfolio_company,initial_acquisition_date,geographic_region,cost,fair_value,asset_class,part,table_index,row_index,shares_units,principal_amount
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,1238736.0,4963620.0,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,2,NaN,NaN
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,46000000.0,61789798.0,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,3,NaN,NaN
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,6324758.0,7742836.0,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,4,NaN,NaN
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,70000000.0,81968442.0,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,5,68333.0,NaN
4,"DFJ Growth V, L.P.",11/27/2024,North America,2475000.0,3683889.0,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,6,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
270,"Term Loan, USD 13.75%, 4/05/2031",4/5/2024,North America,8350486.0,8300529.0,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,13,37,8369421.0,NaN
271,"Delayed Draw, USD 13.75%, 4/05/2031",4/5/2024,North America,4585201.0,4558022.0,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,13,38,4595852.0,NaN
272,"Term Loan, USD 12.75%, 4/05/2031",9/19/2025,North America,4567393.0,4462975.0,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,13,39,4647770.0,NaN
273,"Delayed Draw, USD 12.75%, 4/05/2031",9/19/2025,North America,890339.0,870111.0,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,13,40,906139.0,NaN


In [55]:
# convert cost 
for col in ["cost", "fair_value"]:
    df_investment[col] = (
        df_investment[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        .replace("nan", pd.NA)
    )

    df_investment[col] = pd.to_numeric(df_investment[col], errors="coerce").astype("Int64")

df_investment.drop(columns=["part", "table_index", "row_index"], inplace=True)
df_investment

,portfolio_company,initial_acquisition_date,geographic_region,cost,fair_value,asset_class,shares_units,principal_amount
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,1238736,4963620,Primary Fund Investments — 25.3%,NaN,NaN
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,46000000,61789798,Primary Fund Investments — 25.3%,NaN,NaN
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,6324758,7742836,Primary Fund Investments — 25.3%,NaN,NaN
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,70000000,81968442,Primary Fund Investments — 25.3%,68333.0,NaN
4,"DFJ Growth V, L.P.",11/27/2024,North America,2475000,3683889,Primary Fund Investments — 25.3%,NaN,NaN
...,...,...,...,...,...,...,...,...
276,"RCAF CCG, L.P.",12/29/2025,North America,70000000,70000000,Credit Co-Investments — 3.2%,NaN,NaN
277,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,5061912,5779029,Credit Co-Investments — 3.2%,NaN,NaN
278,Dawson Partners Portfolio Finance Evergreen (A...,8/25/2025,North America,4784,93164,Private Equity — 0.0%,NaN,NaN
279,Cliffwater Corporate Lending Fund — Class I,NaN,North America,100000000,99246704,Registered Funds — 1.8%,NaN,9416196.0


In [56]:
df_investment["profit"] = df_investment["fair_value"] - df_investment["cost"]
df_investment["is_profit"] = df_investment["profit"] > 0
df_investment["is_loss"] = df_investment["profit"] < 0
df_investment["is_breakeven"] = df_investment["profit"] == 0
df_investment

,portfolio_company,initial_acquisition_date,geographic_region,cost,fair_value,asset_class,shares_units,principal_amount,profit,is_profit,is_loss,is_breakeven
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,1238736,4963620,Primary Fund Investments — 25.3%,NaN,NaN,3724884,True,False,False
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,46000000,61789798,Primary Fund Investments — 25.3%,NaN,NaN,15789798,True,False,False
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,6324758,7742836,Primary Fund Investments — 25.3%,NaN,NaN,1418078,True,False,False
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,70000000,81968442,Primary Fund Investments — 25.3%,68333.0,NaN,11968442,True,False,False
4,"DFJ Growth V, L.P.",11/27/2024,North America,2475000,3683889,Primary Fund Investments — 25.3%,NaN,NaN,1208889,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
276,"RCAF CCG, L.P.",12/29/2025,North America,70000000,70000000,Credit Co-Investments — 3.2%,NaN,NaN,0,False,False,True
277,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,5061912,5779029,Credit Co-Investments — 3.2%,NaN,NaN,717117,True,False,False
278,Dawson Partners Portfolio Finance Evergreen (A...,8/25/2025,North America,4784,93164,Private Equity — 0.0%,NaN,NaN,88380,True,False,False
279,Cliffwater Corporate Lending Fund — Class I,NaN,North America,100000000,99246704,Registered Funds — 1.8%,NaN,9416196.0,-753296,False,True,False


In [57]:
df_summary = pd.DataFrame({
    "a": [
        "Total Investment Count",
        "Total Cost", 
        "Total FV", 
        "P&L"
    ],
    "b": [
        len(df_investment),
        int(df_investment["cost"].sum()),
        int(df_investment["fair_value"].sum()),
        int(df_investment["fair_value"].sum())-int(df_investment["cost"].sum()),
    ]
})

df_summary

,a,b
0,Total Investment Count,281
1,Total Cost,4910020304
2,Total FV,5795157385
3,P&L,885137081


In [58]:
# Company-Level Analysis
df_company_level_analysis = (
    df_investment.groupby("portfolio_company")
      .agg(
          total_investments=("portfolio_company", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_company_level_analysis.reset_index(names="Portfolio Company", inplace=True)
df_company_level_analysis["avg_profit"] = round(df_company_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_company_level_analysis["win_rate"] = round(
    df_company_level_analysis["profitable_count"] /
    df_company_level_analysis["total_investments"]
, 2)

df_company_level_analysis["breakeven_rate"] = round(
    df_company_level_analysis["breakeven_count"] /
    df_company_level_analysis["total_investments"]
, 2)

df_company_level_analysis["loss_rate"] = round(
    df_company_level_analysis["loss_count"] /
    df_company_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_company_level_analysis = df_company_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Portfolio Company",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Avg Profit/Investment",
    "Max Profit Investment",
    "Max Loss Investment",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_company_level_analysis = df_company_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_company_level_analysis)

,Portfolio Company,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
201,"Pathway Select Fund, LP — Series A",1,486791324,581325421,94534097,94534097.0,94534097,94534097,1,1.0,0,0.0,0,0.0
200,"Pathway Select Fund, LP — 2025",1,262593295,276914226,14320931,14320931.0,14320931,14320931,1,1.0,0,0.0,0,0.0
240,State Street Institutional U.S. Government Mon...,1,250944884,250944884,0,0.0,0,0,0,0.0,1,1.0,0,0.0
210,"RCAF CCG, L.P.",2,120000000,120000000,0,0.0,0,0,0,0.0,2,1.0,0,0.0
180,"Odyssey Investment Partners Fund VI-A, LP",1,127794621,108878210,-18916411,-18916411.0,-18916411,-18916411,0,0.0,0,0.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,NoHo Holdings III LLC,1,431416,37205,-394211,-394211.0,-394211,-394211,0,0.0,0,0.0,1,1.0
25,"BSP-TS Co-Invest I, LLC",1,5204646,33319,-5171327,-5171327.0,-5171327,-5171327,0,0.0,0,0.0,1,1.0
116,"Incline Equity Partners III, L.P.",1,12165,23157,10992,10992.0,10992,10992,1,1.0,0,0.0,0,0.0
85,"Gores Small Capitalization Partners, L.P.",1,0,17467,0,<NA>,<NA>,<NA>,0,0.0,0,0.0,0,0.0


In [59]:
# Industry-Level Analysis
df_asset_level_analysis = (
    df_investment.groupby("asset_class")
      .agg(
          total_investments=("portfolio_company", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_asset_level_analysis.reset_index(names="asset_class", inplace=True)
df_asset_level_analysis["avg_profit"] = round(df_asset_level_analysis["avg_profit"], 2)


df_asset_level_analysis["win_rate"] = round(
    df_asset_level_analysis["profitable_count"] /
    df_asset_level_analysis["total_investments"]
, 2)

df_asset_level_analysis["breakeven_rate"] = round(
    df_asset_level_analysis["breakeven_count"] /
    df_asset_level_analysis["total_investments"]
, 2)

df_asset_level_analysis["loss_rate"] = round(
    df_asset_level_analysis["loss_count"] /
    df_asset_level_analysis["total_investments"]
, 2)


df_asset_level_analysis = df_asset_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %",
    "asset_class": "Asset Class"
}
desired_order = [
    "Asset Class",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Avg Profit/Investment",
    "Max Profit Investment",
    "Max Loss Investment",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_asset_class_level_analysis = df_asset_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_asset_class_level_analysis)

,Asset Class,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
5,Secondary Fund Investments — 40.4%,140,1797750869,2191590097,394278658,2857091.72,63084920,-18916411,97,0.69,0,0.0,41,0.29
1,Equity Co-Investments — 31.5%,91,1444259764,1706886701,262626937,2886010.3,39307387,-5171327,54,0.59,19,0.21,18,0.2
2,Primary Fund Investments — 25.3%,34,1155837221,1373605299,217768078,6404943.47,94534097,-287517,29,0.85,1,0.03,4,0.12
6,Short-Term Investments — 4.6%,1,250944884,250944884,0,0.0,0,0,0,0.0,1,1.0,0,0.0
0,Credit Co-Investments — 3.2%,13,161222782,172790536,11567754,889827.23,3675086,-104418,7,0.54,1,0.08,5,0.38
4,Registered Funds — 1.8%,1,100000000,99246704,-753296,-753296.0,-753296,-753296,0,0.0,0,0.0,1,1.0
3,Private Equity — 0.0%,1,4784,93164,88380,88380.0,88380,88380,1,1.0,0,0.0,0,0.0


In [62]:
asset_list = list(df_asset_class_level_analysis["Asset Class"])
name = [v.split("—")[0].strip() for v in asset_list]
perc = [v.split("—")[1] for v in asset_list]

df_asset_class_level_analysis = df_asset_class_level_analysis.drop(columns=["Asset Class"])
df_asset_class_level_analysis.insert(0, "Assets", name)
df_asset_class_level_analysis.insert(1, "percentage", perc)

df_asset_class_level_analysis

,Assets,percentage,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
5,Secondary Fund Investments,40.4%,140,1797750869,2191590097,394278658,2857091.72,63084920,-18916411,97,0.69,0,0.0,41,0.29
1,Equity Co-Investments,31.5%,91,1444259764,1706886701,262626937,2886010.3,39307387,-5171327,54,0.59,19,0.21,18,0.2
2,Primary Fund Investments,25.3%,34,1155837221,1373605299,217768078,6404943.47,94534097,-287517,29,0.85,1,0.03,4,0.12
6,Short-Term Investments,4.6%,1,250944884,250944884,0,0.0,0,0,0,0.0,1,1.0,0,0.0
0,Credit Co-Investments,3.2%,13,161222782,172790536,11567754,889827.23,3675086,-104418,7,0.54,1,0.08,5,0.38
4,Registered Funds,1.8%,1,100000000,99246704,-753296,-753296.0,-753296,-753296,0,0.0,0,0.0,1,1.0
3,Private Equity,0.0%,1,4784,93164,88380,88380.0,88380,88380,1,1.0,0,0.0,0,0.0


In [64]:
column_mapping = {
    "portfolio_company": "Portfolio Company",
    "initial_acquisition_date": "Initial Acquisition Date",
    "geographic_region": "Geographic Region",
    "shares_units": "Shares Units",
    "cost": "Cost",
    "fair_value": "Fair Value",
    "first_acquisition_date": "First Acquisition Date",
    "asset_class": "Asset Class",
    "industry": "Industry",
    "principal_amount": "Principal Amount",
    "profit": "Profit"
}

desired_order = [
    "Portfolio Company",
    "Initial Acquisition Date",
    "Geographic Region",
    "Principal Amount",
    "Shares Units",
    "Cost",
    "Fair Value",
    "Profit",
    "First Acquisition Date",
    "Asset Class",
    "Industry"
]
df_investment_formatted = df_investment.rename(columns=column_mapping).reindex(columns=desired_order)
df_investment_formatted

,Portfolio Company,Initial Acquisition Date,Geographic Region,Principal Amount,Shares Units,Cost,Fair Value,Profit,First Acquisition Date,Asset Class,Industry
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,NaN,NaN,1238736,4963620,3724884,NaN,Primary Fund Investments — 25.3%,NaN
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,NaN,NaN,46000000,61789798,15789798,NaN,Primary Fund Investments — 25.3%,NaN
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,NaN,NaN,6324758,7742836,1418078,NaN,Primary Fund Investments — 25.3%,NaN
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,NaN,68333.0,70000000,81968442,11968442,NaN,Primary Fund Investments — 25.3%,NaN
4,"DFJ Growth V, L.P.",11/27/2024,North America,NaN,NaN,2475000,3683889,1208889,NaN,Primary Fund Investments — 25.3%,NaN
...,...,...,...,...,...,...,...,...,...,...,...
276,"RCAF CCG, L.P.",12/29/2025,North America,NaN,NaN,70000000,70000000,0,NaN,Credit Co-Investments — 3.2%,NaN
277,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,NaN,NaN,5061912,5779029,717117,NaN,Credit Co-Investments — 3.2%,NaN
278,Dawson Partners Portfolio Finance Evergreen (A...,8/25/2025,North America,NaN,NaN,4784,93164,88380,NaN,Private Equity — 0.0%,NaN
279,Cliffwater Corporate Lending Fund — Class I,NaN,North America,9416196.0,NaN,100000000,99246704,-753296,NaN,Registered Funds — 1.8%,NaN


### Dumping to GS

In [66]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "service_account.json",
    scopes=SCOPES
)

client = gspread.authorize(creds)

In [67]:
df_map = {
    "Summary": df_summary,
    "All Investments": df_investment_formatted,
    "Asset Level Analysis": df_asset_class_level_analysis,
    "Company Level Analysis": df_company_level_analysis
}

In [68]:
spreadsheet = client.open("CPEFX-NCSR-20260331")

In [69]:
for tab_name, df_analysis in df_map.items():
    try:
        worksheet = spreadsheet.worksheet(tab_name)
        worksheet.clear()
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=max(len(df_analysis) + 10, 100),
            cols=max(len(df_analysis.columns) + 10, 20)
        )

    display(df_analysis)
    set_with_dataframe(worksheet, df_analysis)
    
    header_fmt = CellFormat(
        backgroundColor=Color(0.85, 0.90, 1.0),
        textFormat=TextFormat(bold=True)
    )

    format_cell_range(
        worksheet,
        "1:1",
        header_fmt
    )

    worksheet.freeze(rows=1)
    worksheet.columns_auto_resize(0, len(df_analysis.columns))

,a,b
0,Total Investment Count,281
1,Total Cost,4910020304
2,Total FV,5795157385
3,P&L,885137081


,Portfolio Company,Initial Acquisition Date,Geographic Region,Principal Amount,Shares Units,Cost,Fair Value,Profit,First Acquisition Date,Asset Class,Industry
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,NaN,NaN,1238736,4963620,3724884,NaN,Primary Fund Investments — 25.3%,NaN
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,NaN,NaN,46000000,61789798,15789798,NaN,Primary Fund Investments — 25.3%,NaN
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,NaN,NaN,6324758,7742836,1418078,NaN,Primary Fund Investments — 25.3%,NaN
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,NaN,68333.0,70000000,81968442,11968442,NaN,Primary Fund Investments — 25.3%,NaN
4,"DFJ Growth V, L.P.",11/27/2024,North America,NaN,NaN,2475000,3683889,1208889,NaN,Primary Fund Investments — 25.3%,NaN
...,...,...,...,...,...,...,...,...,...,...,...
276,"RCAF CCG, L.P.",12/29/2025,North America,NaN,NaN,70000000,70000000,0,NaN,Credit Co-Investments — 3.2%,NaN
277,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,NaN,NaN,5061912,5779029,717117,NaN,Credit Co-Investments — 3.2%,NaN
278,Dawson Partners Portfolio Finance Evergreen (A...,8/25/2025,North America,NaN,NaN,4784,93164,88380,NaN,Private Equity — 0.0%,NaN
279,Cliffwater Corporate Lending Fund — Class I,NaN,North America,9416196.0,NaN,100000000,99246704,-753296,NaN,Registered Funds — 1.8%,NaN


,Assets,percentage,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
5,Secondary Fund Investments,40.4%,140,1797750869,2191590097,394278658,2857091.72,63084920,-18916411,97,0.69,0,0.0,41,0.29
1,Equity Co-Investments,31.5%,91,1444259764,1706886701,262626937,2886010.3,39307387,-5171327,54,0.59,19,0.21,18,0.2
2,Primary Fund Investments,25.3%,34,1155837221,1373605299,217768078,6404943.47,94534097,-287517,29,0.85,1,0.03,4,0.12
6,Short-Term Investments,4.6%,1,250944884,250944884,0,0.0,0,0,0,0.0,1,1.0,0,0.0
0,Credit Co-Investments,3.2%,13,161222782,172790536,11567754,889827.23,3675086,-104418,7,0.54,1,0.08,5,0.38
4,Registered Funds,1.8%,1,100000000,99246704,-753296,-753296.0,-753296,-753296,0,0.0,0,0.0,1,1.0
3,Private Equity,0.0%,1,4784,93164,88380,88380.0,88380,88380,1,1.0,0,0.0,0,0.0


,Portfolio Company,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
201,"Pathway Select Fund, LP — Series A",1,486791324,581325421,94534097,94534097.0,94534097,94534097,1,1.0,0,0.0,0,0.0
200,"Pathway Select Fund, LP — 2025",1,262593295,276914226,14320931,14320931.0,14320931,14320931,1,1.0,0,0.0,0,0.0
240,State Street Institutional U.S. Government Mon...,1,250944884,250944884,0,0.0,0,0,0,0.0,1,1.0,0,0.0
210,"RCAF CCG, L.P.",2,120000000,120000000,0,0.0,0,0,0,0.0,2,1.0,0,0.0
180,"Odyssey Investment Partners Fund VI-A, LP",1,127794621,108878210,-18916411,-18916411.0,-18916411,-18916411,0,0.0,0,0.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,NoHo Holdings III LLC,1,431416,37205,-394211,-394211.0,-394211,-394211,0,0.0,0,0.0,1,1.0
25,"BSP-TS Co-Invest I, LLC",1,5204646,33319,-5171327,-5171327.0,-5171327,-5171327,0,0.0,0,0.0,1,1.0
116,"Incline Equity Partners III, L.P.",1,12165,23157,10992,10992.0,10992,10992,1,1.0,0,0.0,0,0.0
85,"Gores Small Capitalization Partners, L.P.",1,0,17467,0,<NA>,<NA>,<NA>,0,0.0,0,0.0,0,0.0


In [33]:
for i  in df_investment['asset_class'].unique(): print(i)

Primary Fund Investments — 25.3%
Secondary Fund Investments — 40.4%
Equity Co-Investments — 31.5%
Credit Co-Investments — 3.2%
Private Equity — 0.0%
Registered Funds — 1.8%
Short-Term Investments — 4.6%


nan


,portfolio_company,initial_acquisition_date,geographic_region,cost,fair_value,asset_class,industry,part,table_index,row_index,shares_units,principal_amount
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,1238736.0,4963620.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,2,NaN,NaN
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,46000000.0,61789798.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,3,NaN,NaN
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,6324758.0,7742836.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,4,NaN,NaN
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,70000000.0,81968442.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,5,68333.0,NaN
4,"DFJ Growth V, L.P.",11/27/2024,North America,2475000.0,3683889.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,6,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
276,"RCAF CCG, L.P.",12/29/2025,North America,70000000.0,70000000.0,NaN,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,15,2,NaN,NaN
277,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,5061912.0,5779029.0,NaN,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,15,3,NaN,NaN
278,Dawson Partners Portfolio Finance Evergreen (A...,8/25/2025,North America,4784.0,93164.0,NaN,Private Equity — 0.0%,ea0289743-01_ncsr,15,7,NaN,NaN
279,Cliffwater Corporate Lending Fund — Class I,NaN,North America,100000000.0,99246704.0,NaN,Registered Funds — 1.8%,ea0289743-01_ncsr,15,11,NaN,9416196.0
